# Model Training - Water Potability

## Thành viên 2

Các mô hình phụ trách:

1. Logistic Regression
2. Support Vector Machine (SVM)

Trong giai đoạn này:

- Sử dụng cùng Train/Test Split của toàn bộ nhóm.
- `random_state = 42`.
- `stratify = y`.
- Missing values được xử lý bằng Median Imputation.
- Logistic Regression và SVM sử dụng StandardScaler.
- Hyperparameter tuning chỉ thực hiện trên tập train.
- Test set chưa được sử dụng để lựa chọn model hoặc tham số.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold

In [2]:
project_root = Path.cwd()

# Nếu notebook chạy trong ai-models/colab
if project_root.name == "colab":
    project_root = project_root.parents[1]

src_path = project_root / "ai-models" / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from preprocess import (
    load_dataset,
    split_features_target,
    split_train_test,
    build_scaled_preprocessor,
    RANDOM_STATE,
)

print("Import preprocessing thành công.")
print("Random state:", RANDOM_STATE)

Import preprocessing thành công.
Random state: 42


In [3]:
data_path = project_root / "ai-models" / "data" / "water_potability.csv"

df = load_dataset(data_path)

print("Đọc dataset thành công.")
print("Dataset shape:", df.shape)
print("Dataset path:", data_path)

Đọc dataset thành công.
Dataset shape: (3276, 10)
Dataset path: f:\4\Hoc may co ban\WaterQualityProject\ai-models\data\water_potability.csv


In [4]:
X, y = split_features_target(df)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (3276, 9)
y shape: (3276,)


In [5]:
X_train, X_test, y_train, y_test = split_train_test(X, y)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (2620, 9)
X_test : (656, 9)
y_train: (2620,)
y_test : (656,)


In [6]:
def show_class_distribution(series, name):
    counts = series.value_counts().sort_index()
    percents = (
        series.value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )

    result = pd.DataFrame({
        "Count": counts,
        "Percent": percents
    })

    print(name)
    display(result)


show_class_distribution(y_train, "Training set")
show_class_distribution(y_test, "Test set")

Training set


,Count,Percent
Potability,,
0,1598,60.99
1,1022,39.01


Test set


,Count,Percent
Potability,,
0,400,60.98
1,256,39.02


In [7]:
print("Train samples:", len(X_train))
print("Test samples :", len(X_test))

print(
    "Train/Test overlap:",
    len(X_train.index.intersection(X_test.index))
)

Train samples: 2620
Test samples : 656
Train/Test overlap: 0


## Chuẩn bị dữ liệu huấn luyện

Dataset được chia bằng hàm `split_train_test()` dùng chung trong
`ai-models/src/preprocess.py`.

Cấu hình:

- Training set: 80%.
- Test set: 20%.
- `random_state = 42`.
- `stratify = y`.

Kết quả:

- Training set: 2620 mẫu.
- Test set: 656 mẫu.
- Hai tập không có index trùng nhau.
- Tỷ lệ hai lớp được duy trì gần giống nhau nhờ `stratify`.

Tập test được giữ riêng cho bước đánh giá cuối cùng.
Hyperparameter tuning chỉ được thực hiện trên training set.

## 1. Logistic Regression - Baseline Model

Logistic Regression được sử dụng làm một mô hình phân loại cơ bản để so sánh
với các mô hình khác trong project.

Pipeline gồm:

1. Median Imputation để xử lý missing values.
2. StandardScaler để chuẩn hóa các feature.
3. Logistic Regression để thực hiện phân loại.

Toàn bộ preprocessing được đặt bên trong Pipeline để đảm bảo các tham số
như median, mean và standard deviation chỉ được học từ dữ liệu training.

Tập test chưa được sử dụng trong quá trình lựa chọn hoặc tinh chỉnh model.

In [8]:
logistic_pipeline = Pipeline([
    (
        "preprocessor",
        build_scaled_preprocessor()
    ),
    (
        "classifier",
        LogisticRegression(
            random_state=RANDOM_STATE,
            max_iter=2000
        )
    )
])

logistic_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each 

In [9]:
from sklearn.utils.validation import check_is_fitted
from sklearn.exceptions import NotFittedError

try:
    check_is_fitted(logistic_pipeline)
    print("Logistic Pipeline đã được fit.")
except NotFittedError:
    print("Logistic Pipeline chưa được fit - đúng yêu cầu.")

Logistic Pipeline chưa được fit - đúng yêu cầu.


In [10]:
logistic_pipeline.fit(
    X_train,
    y_train
)

print("Huấn luyện Logistic Regression baseline thành công.")

Huấn luyện Logistic Regression baseline thành công.


In [11]:
try:
    check_is_fitted(logistic_pipeline)
    print("Logistic Pipeline đã được fit thành công.")
except NotFittedError:
    print("Pipeline chưa được fit.")

Logistic Pipeline đã được fit thành công.


In [12]:
fitted_preprocessor = logistic_pipeline.named_steps["preprocessor"]

fitted_imputer = fitted_preprocessor.named_steps["imputer"]

imputer_statistics = pd.Series(
    fitted_imputer.statistics_,
    index=X_train.columns,
    name="Train_Median"
)

imputer_statistics

ph                     7.035037
Hardness             196.928061
Solids             20866.335842
Chloramines            7.118162
Sulfate              333.073546
Conductivity         424.941336
Organic_carbon        14.214780
Trihalomethanes       66.565709
Turbidity              3.969602
Name: Train_Median, dtype: float64

In [13]:
fitted_scaler = fitted_preprocessor.named_steps["scaler"]

scaler_statistics = pd.DataFrame({
    "Train_Mean": fitted_scaler.mean_,
    "Train_Scale": fitted_scaler.scale_
}, index=X_train.columns)

scaler_statistics.round(4)

,Train_Mean,Train_Scale
ph,7.0800,1.4695
Hardness,196.5208,32.6315
Solids,21888.0677,8758.4374
Chloramines,7.1164,1.5989
Sulfate,333.7697,36.2292
Conductivity,427.9159,80.9284
Organic_carbon,14.2727,3.2959
Trihalomethanes,66.1643,15.7922
Turbidity,3.9738,0.7802


### Kết quả Logistic Regression Baseline

Logistic Regression đã được huấn luyện thành công trên training set.

Preprocessing được thực hiện hoàn toàn bên trong Pipeline:

`Median Imputation`
→ `StandardScaler`
→ `Logistic Regression`

Các tham số preprocessing chỉ được học từ `X_train`.

Tập test vẫn được giữ riêng và chưa được sử dụng để lựa chọn model
hoặc hyperparameter.

Bước tiếp theo là đánh giá Logistic Regression bằng Cross-Validation
trên training set trước khi thực hiện Hyperparameter Tuning.

## 2. Cross-Validation - Logistic Regression Baseline

Để đánh giá độ ổn định của Logistic Regression trước khi tuning,
mô hình được đánh giá bằng Stratified K-Fold Cross-Validation
trên training set.

Cấu hình:

- 5 folds.
- `shuffle=True`.
- `random_state=42`.
- Chỉ sử dụng `X_train` và `y_train`.
- Không sử dụng test set.

Các metric được theo dõi:

- Accuracy.
- Precision.
- Recall.
- F1-score.
- ROC-AUC.

Ngoài giá trị trung bình, độ lệch chuẩn giữa các fold cũng được xem xét
để đánh giá mức độ ổn định của mô hình.

In [14]:
from sklearn.model_selection import cross_validate
from sklearn.base import clone

In [15]:
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_strategy

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

In [16]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

scoring

{'accuracy': 'accuracy',
 'precision': 'precision',
 'recall': 'recall',
 'f1': 'f1',
 'roc_auc': 'roc_auc'}

In [17]:
logistic_cv_pipeline = clone(logistic_pipeline)

logistic_cv_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each 

In [18]:
logistic_cv_results = cross_validate(
    estimator=logistic_cv_pipeline,
    X=X_train,
    y=y_train,
    cv=cv_strategy,
    scoring=scoring,
    return_train_score=True
)

print("Cross-Validation Logistic Regression hoàn thành.")

Cross-Validation Logistic Regression hoàn thành.


f:\4\Hoc may co ban\WaterQualityProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\4\Hoc may co ban\WaterQualityProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\4\Hoc may co ban\WaterQualityProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result

In [19]:
cv_detail = pd.DataFrame({
    "Fold": range(1, 6),
    "Accuracy": logistic_cv_results["test_accuracy"],
    "Precision": logistic_cv_results["test_precision"],
    "Recall": logistic_cv_results["test_recall"],
    "F1": logistic_cv_results["test_f1"],
    "ROC_AUC": logistic_cv_results["test_roc_auc"]
})

cv_detail.round(4)

,Fold,Accuracy,Precision,Recall,F1,ROC_AUC
0,1,0.6107,0.0,0.0,0.0,0.4764
1,2,0.6107,0.0,0.0,0.0,0.4809
2,3,0.6107,0.0,0.0,0.0,0.4856
3,4,0.6088,0.0,0.0,0.0,0.4988
4,5,0.6088,0.0,0.0,0.0,0.4453


In [20]:
cv_summary = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC"
    ],
    "Mean": [
        logistic_cv_results["test_accuracy"].mean(),
        logistic_cv_results["test_precision"].mean(),
        logistic_cv_results["test_recall"].mean(),
        logistic_cv_results["test_f1"].mean(),
        logistic_cv_results["test_roc_auc"].mean()
    ],
    "Std": [
        logistic_cv_results["test_accuracy"].std(),
        logistic_cv_results["test_precision"].std(),
        logistic_cv_results["test_recall"].std(),
        logistic_cv_results["test_f1"].std(),
        logistic_cv_results["test_roc_auc"].std()
    ]
})

cv_summary.round(4)

,Metric,Mean,Std
0,Accuracy,0.6099,0.0009
1,Precision,0.0000,0.0000
2,Recall,0.0000,0.0000
3,F1,0.0000,0.0000
4,ROC_AUC,0.4774,0.0177


In [21]:
train_validation_comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC"
    ],
    "Train_Mean": [
        logistic_cv_results["train_accuracy"].mean(),
        logistic_cv_results["train_precision"].mean(),
        logistic_cv_results["train_recall"].mean(),
        logistic_cv_results["train_f1"].mean(),
        logistic_cv_results["train_roc_auc"].mean()
    ],
    "Validation_Mean": [
        logistic_cv_results["test_accuracy"].mean(),
        logistic_cv_results["test_precision"].mean(),
        logistic_cv_results["test_recall"].mean(),
        logistic_cv_results["test_f1"].mean(),
        logistic_cv_results["test_roc_auc"].mean()
    ]
})

train_validation_comparison["Gap"] = (
    train_validation_comparison["Train_Mean"]
    - train_validation_comparison["Validation_Mean"]
)

train_validation_comparison.round(4)

,Metric,Train_Mean,Validation_Mean,Gap
0,Accuracy,0.6103,0.6099,0.0004
1,Precision,0.4000,0.0000,0.4000
2,Recall,0.0010,0.0000,0.0010
3,F1,0.0020,0.0000,0.0020
4,ROC_AUC,0.5216,0.4774,0.0442


### Nhận xét Logistic Regression Baseline

Kết quả Cross-Validation cho thấy Logistic Regression baseline chưa phân loại
tốt lớp `Potability = 1`.

- Validation Accuracy đạt khoảng 0.6099.
- Validation Precision = 0.
- Validation Recall = 0.
- Validation F1-score = 0.
- Validation ROC-AUC khoảng 0.4774.

Accuracy khoảng 61% gần bằng chính tỷ lệ của lớp `Potability = 0`
trong dataset. Vì vậy Accuracy cao hơn 0.5 không đồng nghĩa mô hình đang
phân loại tốt.

Recall và F1 bằng 0 cho thấy mô hình gần như không dự đoán được
các mẫu thuộc lớp `Potability = 1`.

ROC-AUC cũng gần mức 0.5, cho thấy khả năng phân biệt hai lớp của
Logistic Regression baseline còn yếu.

Khoảng cách giữa Accuracy train và validation rất nhỏ nên chưa cho thấy
dấu hiệu overfitting rõ ràng. Vấn đề chính của baseline hiện tại là
khả năng học ranh giới phân loại còn hạn chế và xu hướng nghiêng về lớp 0.

Do đó không lựa chọn Logistic Regression baseline làm cấu hình cuối.
Bước tiếp theo là thực hiện Hyperparameter Tuning chỉ trên training set,
đặc biệt xem xét tham số `C` và `class_weight`, sau đó đánh giá lại bằng
Cross-Validation.

Tập test vẫn chưa được sử dụng.

## 3. Hyperparameter Tuning - Logistic Regression

Logistic Regression baseline có xu hướng nghiêng mạnh về lớp 0.

Hyperparameter tuning được thực hiện trên training set bằng GridSearchCV.

Các tham số được khảo sát:

- `C`: mức độ regularization.
- `class_weight`: xem xét cân bằng ảnh hưởng của hai lớp.
- `solver`: thuật toán tối ưu.

Metric chính dùng để lựa chọn cấu hình trong GridSearchCV là `f1`
nhằm cân bằng Precision và Recall của lớp `Potability = 1`.

Test set không được sử dụng trong quá trình tuning.

In [23]:
logistic_tuning_pipeline = Pipeline([
    (
        "preprocessor",
        build_scaled_preprocessor()
    ),
    (
        "classifier",
        LogisticRegression(
            random_state=RANDOM_STATE,
            max_iter=2000
        )
    )
])

logistic_tuning_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each 

In [24]:
logistic_param_grid = {
    "classifier__C": [
        0.01,
        0.1,
        1.0,
        10.0,
        100.0
    ],
    "classifier__class_weight": [
        None,
        "balanced"
    ],
    "classifier__solver": [
        "liblinear",
        "lbfgs"
    ]
}

logistic_param_grid

{'classifier__C': [0.01, 0.1, 1.0, 10.0, 100.0],
 'classifier__class_weight': [None, 'balanced'],
 'classifier__solver': ['liblinear', 'lbfgs']}

In [25]:
logistic_grid_search = GridSearchCV(
    estimator=logistic_tuning_pipeline,
    param_grid=logistic_param_grid,
    scoring="f1",
    cv=cv_strategy,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

logistic_grid_search

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__C': [0.01, 0.1, ...], 'classifier__class_weight': [None, 'balanced'], 'classifier__solver': ['liblinear', 'lbfgs']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` 

In [27]:
logistic_grid_search.fit(
    X_train,
    y_train
)

print("GridSearchCV Logistic Regression hoàn thành.")

Fitting 5 folds for each of 20 candidates, totalling 100 fits
GridSearchCV Logistic Regression hoàn thành.


In [28]:
print("Best parameters:")
print(logistic_grid_search.best_params_)

print("\nBest CV F1:")
print(round(logistic_grid_search.best_score_, 4))

Best parameters:
{'classifier__C': 0.01, 'classifier__class_weight': 'balanced', 'classifier__solver': 'liblinear'}

Best CV F1:
0.4183


In [29]:
logistic_grid_results = pd.DataFrame(
    logistic_grid_search.cv_results_
)

logistic_top_results = (
    logistic_grid_results[
        [
            "param_classifier__C",
            "param_classifier__class_weight",
            "param_classifier__solver",
            "mean_train_score",
            "mean_test_score",
            "std_test_score",
            "rank_test_score"
        ]
    ]
    .sort_values("rank_test_score")
    .head(10)
)

logistic_top_results

,param_classifier__C,param_classifier__class_weight,param_classifier__solver,mean_train_score,mean_test_score,std_test_score,rank_test_score
2,0.01,balanced,liblinear,0.446192,0.418330,0.028831,1
3,0.01,balanced,lbfgs,0.445518,0.417911,0.029995,2
6,0.10,balanced,liblinear,0.445853,0.416432,0.028459,3
10,1.00,balanced,liblinear,0.445418,0.415718,0.028433,4
14,10.00,balanced,liblinear,0.445638,0.415718,0.028433,4
18,100.00,balanced,liblinear,0.445638,0.415718,0.028433,4
7,0.10,balanced,lbfgs,0.445591,0.415702,0.028218,7
11,1.00,balanced,lbfgs,0.445734,0.415525,0.028335,8
19,100.00,balanced,lbfgs,0.445903,0.415525,0.028335,8
15,10.00,balanced,lbfgs,0.445903,0.415525,0.028335,8


## 4. Đánh giá Logistic Regression sau Hyperparameter Tuning

GridSearchCV tìm được cấu hình tốt nhất:

- `C = 0.01`
- `class_weight = "balanced"`
- `solver = "liblinear"`

Best Cross-Validation F1 đạt khoảng `0.4183`.

So với Logistic Regression baseline, cấu hình mới cải thiện khả năng
nhận diện lớp `Potability = 1`.

Tuy nhiên, model chưa được đánh giá trên test set ở bước này.
Tiếp tục sử dụng Cross-Validation trên training set để xem đầy đủ:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- Train/Validation gap

In [30]:
best_logistic_pipeline = logistic_grid_search.best_estimator_

best_logistic_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['ph','Hardness','Solids',...,'Organic_carbon','Trihalomethanes', 'Turbidity']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be insp

In [34]:
scoring_metrics = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

scoring_metrics

{'accuracy': 'accuracy',
 'precision': 'precision',
 'recall': 'recall',
 'f1': 'f1',
 'roc_auc': 'roc_auc'}

In [36]:
best_logistic_cv_results = cross_validate(
    estimator=best_logistic_pipeline,
    X=X_train,
    y=y_train,
    cv=cv_strategy,
    scoring=scoring_metrics,
    return_train_score=True
)

print("Cross-Validation best Logistic Regression hoàn thành.")

Cross-Validation best Logistic Regression hoàn thành.


In [37]:
best_logistic_cv_results.keys()

dict_keys(['fit_time', 'score_time', 'test_accuracy', 'train_accuracy', 'test_precision', 'train_precision', 'test_recall', 'train_recall', 'test_f1', 'train_f1', 'test_roc_auc', 'train_roc_auc'])

In [38]:
best_logistic_cv_detail = pd.DataFrame({
    "Fold": range(1, 6),
    "Accuracy": best_logistic_cv_results["test_accuracy"],
    "Precision": best_logistic_cv_results["test_precision"],
    "Recall": best_logistic_cv_results["test_recall"],
    "F1": best_logistic_cv_results["test_f1"],
    "ROC_AUC": best_logistic_cv_results["test_roc_auc"]
})

best_logistic_cv_detail.round(4)

,Fold,Accuracy,Precision,Recall,F1,ROC_AUC
0,1,0.4885,0.3699,0.4461,0.4044,0.4754
1,2,0.5019,0.3978,0.5441,0.4596,0.4804
2,3,0.5134,0.3959,0.4755,0.4321,0.4861
3,4,0.5095,0.3917,0.4585,0.4225,0.4968
4,5,0.4676,0.3458,0.4049,0.3730,0.4442


In [39]:
best_logistic_cv_summary = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC"
    ],
    "Mean": [
        best_logistic_cv_results["test_accuracy"].mean(),
        best_logistic_cv_results["test_precision"].mean(),
        best_logistic_cv_results["test_recall"].mean(),
        best_logistic_cv_results["test_f1"].mean(),
        best_logistic_cv_results["test_roc_auc"].mean()
    ],
    "Std": [
        best_logistic_cv_results["test_accuracy"].std(),
        best_logistic_cv_results["test_precision"].std(),
        best_logistic_cv_results["test_recall"].std(),
        best_logistic_cv_results["test_f1"].std(),
        best_logistic_cv_results["test_roc_auc"].std()
    ]
})

best_logistic_cv_summary.round(4)

,Metric,Mean,Std
0,Accuracy,0.4962,0.0166
1,Precision,0.3802,0.0199
2,Recall,0.4658,0.0456
3,F1,0.4183,0.0288
4,ROC_AUC,0.4766,0.0177


In [41]:
best_logistic_train_validation = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC"
    ],
    "Train_Mean": [
        best_logistic_cv_results["train_accuracy"].mean(),
        best_logistic_cv_results["train_precision"].mean(),
        best_logistic_cv_results["train_recall"].mean(),
        best_logistic_cv_results["train_f1"].mean(),
        best_logistic_cv_results["train_roc_auc"].mean()
    ],
    "Validation_Mean": [
        best_logistic_cv_results["test_accuracy"].mean(),
        best_logistic_cv_results["test_precision"].mean(),
        best_logistic_cv_results["test_recall"].mean(),
        best_logistic_cv_results["test_f1"].mean(),
        best_logistic_cv_results["test_roc_auc"].mean()
    ]
})

best_logistic_train_validation["Gap"] = (
    best_logistic_train_validation["Train_Mean"]
    - best_logistic_train_validation["Validation_Mean"]
)

best_logistic_train_validation.round(4)

,Metric,Train_Mean,Validation_Mean,Gap
0,Accuracy,0.5147,0.4962,0.0185
1,Precision,0.4021,0.3802,0.0218
2,Recall,0.5012,0.4658,0.0354
3,F1,0.4462,0.4183,0.0279
4,ROC_AUC,0.5220,0.4766,0.0454


### Kết luận Logistic Regression

Logistic Regression baseline có xu hướng dự đoán phần lớn mẫu về lớp
`Potability = 0`.

Ở baseline:

- Validation Accuracy khoảng 0.6099.
- Validation Recall = 0.
- Validation F1 = 0.
- Validation ROC-AUC khoảng 0.4774.

Sau Hyperparameter Tuning, GridSearchCV lựa chọn:

- `C = 0.01`
- `class_weight = "balanced"`
- `solver = "liblinear"`

Kết quả Cross-Validation của cấu hình sau tuning:

- Accuracy: 0.4962 ± 0.0166.
- Precision: 0.3802 ± 0.0199.
- Recall: 0.4658 ± 0.0456.
- F1-score: 0.4183 ± 0.0288.
- ROC-AUC: 0.4766 ± 0.0177.

Việc sử dụng `class_weight="balanced"` giúp mô hình quan tâm nhiều hơn
đến lớp `Potability = 1`. Recall và F1 được cải thiện rõ rệt so với
baseline, thay vì gần như không phát hiện được lớp 1.

Tuy nhiên Accuracy giảm và ROC-AUC vẫn thấp, cho thấy Logistic Regression
còn hạn chế trong việc mô hình hóa quan hệ giữa các đặc trưng chất lượng
nước và biến mục tiêu.

Khoảng cách giữa Train và Validation của các metric tương đối nhỏ,
do đó chưa có dấu hiệu overfitting nghiêm trọng. Hạn chế chính hiện tại
là khả năng phân biệt hai lớp của mô hình.

Kết quả trên chỉ được sử dụng trong quá trình phát triển mô hình.
Test set vẫn chưa được sử dụng và sẽ được giữ lại cho bước đánh giá cuối cùng.

# 5. Support Vector Machine - SVM

Mô hình thứ hai của thành viên 2 là Support Vector Machine.

SVM phù hợp để kiểm tra khả năng phân loại với ranh giới phi tuyến,
đặc biệt thông qua RBF Kernel.

Pipeline của SVM gồm:

1. Median Imputation.
2. StandardScaler.
3. Support Vector Classifier.

StandardScaler là bước quan trọng đối với SVM vì thuật toán nhạy cảm
với thang đo của các feature.

Trước tiên, mô hình SVM mặc định được sử dụng làm baseline.
Sau đó mới thực hiện Hyperparameter Tuning trên training set.

Test set chưa được sử dụng.

In [42]:
from sklearn.svm import SVC

print("Import SVC thành công.")

Import SVC thành công.


In [43]:
svm_baseline_pipeline = Pipeline([
    (
        "preprocessor",
        build_scaled_preprocessor()
    ),
    (
        "classifier",
        SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale"
        )
    )
])

svm_baseline_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each 

In [44]:
svm_baseline_pipeline.fit(
    X_train,
    y_train
)

print("Huấn luyện SVM baseline thành công.")

Huấn luyện SVM baseline thành công.


In [45]:
cv_strategy
scoring_metrics

{'accuracy': 'accuracy',
 'precision': 'precision',
 'recall': 'recall',
 'f1': 'f1',
 'roc_auc': 'roc_auc'}

In [47]:
svm_cv_results = cross_validate(
    estimator=svm_baseline_pipeline,
    X=X_train,
    y=y_train,
    cv=cv_strategy,
    scoring=scoring_metrics,
    return_train_score=True
)

print("Cross-Validation SVM baseline hoàn thành.")

Cross-Validation SVM baseline hoàn thành.


In [48]:
svm_cv_detail = pd.DataFrame({
    "Fold": range(1, 6),
    "Accuracy": svm_cv_results["test_accuracy"],
    "Precision": svm_cv_results["test_precision"],
    "Recall": svm_cv_results["test_recall"],
    "F1": svm_cv_results["test_f1"],
    "ROC_AUC": svm_cv_results["test_roc_auc"]
})

svm_cv_detail.round(4)

,Fold,Accuracy,Precision,Recall,F1,ROC_AUC
0,1,0.6966,0.7647,0.3186,0.4498,0.7157
1,2,0.6737,0.6988,0.2843,0.4042,0.7020
2,3,0.6775,0.7160,0.2843,0.4070,0.6900
3,4,0.6870,0.7113,0.3366,0.4570,0.7263
4,5,0.6641,0.7042,0.2439,0.3623,0.6757


In [49]:
svm_cv_summary = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC"
    ],
    "Mean": [
        svm_cv_results["test_accuracy"].mean(),
        svm_cv_results["test_precision"].mean(),
        svm_cv_results["test_recall"].mean(),
        svm_cv_results["test_f1"].mean(),
        svm_cv_results["test_roc_auc"].mean()
    ],
    "Std": [
        svm_cv_results["test_accuracy"].std(),
        svm_cv_results["test_precision"].std(),
        svm_cv_results["test_recall"].std(),
        svm_cv_results["test_f1"].std(),
        svm_cv_results["test_roc_auc"].std()
    ]
})

svm_cv_summary.round(4)

,Metric,Mean,Std
0,Accuracy,0.6798,0.0112
1,Precision,0.7190,0.0236
2,Recall,0.2935,0.0320
3,F1,0.4161,0.0344
4,ROC_AUC,0.7019,0.0179


In [50]:
svm_train_validation = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC"
    ],
    "Train_Mean": [
        svm_cv_results["train_accuracy"].mean(),
        svm_cv_results["train_precision"].mean(),
        svm_cv_results["train_recall"].mean(),
        svm_cv_results["train_f1"].mean(),
        svm_cv_results["train_roc_auc"].mean()
    ],
    "Validation_Mean": [
        svm_cv_results["test_accuracy"].mean(),
        svm_cv_results["test_precision"].mean(),
        svm_cv_results["test_recall"].mean(),
        svm_cv_results["test_f1"].mean(),
        svm_cv_results["test_roc_auc"].mean()
    ]
})

svm_train_validation["Gap"] = (
    svm_train_validation["Train_Mean"]
    - svm_train_validation["Validation_Mean"]
)

svm_train_validation.round(4)

,Metric,Train_Mean,Validation_Mean,Gap
0,Accuracy,0.7409,0.6798,0.0612
1,Precision,0.8683,0.7190,0.1493
2,Recall,0.3960,0.2935,0.1025
3,F1,0.5438,0.4161,0.1277
4,ROC_AUC,0.8225,0.7019,0.1206


### Kết luận Support Vector Machine (SVM)

SVM được xây dựng theo Pipeline gồm:

`Median Imputation`
→ `StandardScaler`
→ `SVC`

Mô hình baseline sử dụng:

- `kernel = "rbf"`
- `C = 1.0`
- `gamma = "scale"`

SVM được đánh giá bằng Stratified 5-Fold Cross-Validation trên training set.
Test set chưa được sử dụng trong quá trình huấn luyện và đánh giá Cross-Validation.

Kết quả Cross-Validation của SVM baseline:

- Accuracy: `0.6798 ± 0.0112`
- Precision: `0.7190 ± 0.0236`
- Recall: `0.2935 ± 0.0320`
- F1-score: `0.4161 ± 0.0344`
- ROC-AUC: `0.7019 ± 0.0179`

SVM baseline đạt Accuracy và Precision tương đối cao.
Precision = 0.7190 cho thấy khi mô hình dự đoán một mẫu thuộc lớp
`Potability = 1`, tỷ lệ dự đoán đúng tương đối tốt.

Tuy nhiên Recall chỉ đạt 0.2935, nghĩa là mô hình vẫn bỏ sót khá nhiều
mẫu thực sự thuộc lớp `Potability = 1`.

F1-score đạt 0.4161, phản ánh sự mất cân bằng giữa Precision cao và
Recall thấp.

ROC-AUC đạt 0.7019, cao hơn đáng kể so với Logistic Regression,
cho thấy SVM có khả năng phân biệt hai lớp tốt hơn trong Cross-Validation.

Kết quả Train và Validation:

- Accuracy gap: `0.0612`
- Precision gap: `0.1493`
- Recall gap: `0.1025`
- F1 gap: `0.1277`
- ROC-AUC gap: `0.1206`

Các khoảng cách Train–Validation của Precision, Recall, F1 và ROC-AUC
khá lớn hơn so với Logistic Regression. Điều này cho thấy SVM baseline
có dấu hiệu overfitting nhất định: mô hình hoạt động tốt hơn rõ rệt
trên training data so với validation data.

So với Logistic Regression sau tuning, SVM baseline có:

- Accuracy cao hơn.
- Precision cao hơn.
- ROC-AUC cao hơn.
- Recall thấp hơn.
- F1-score gần tương đương.

Nhìn chung, SVM thể hiện khả năng phân biệt hai lớp tốt hơn Logistic Regression,
nhưng khả năng phát hiện đầy đủ các mẫu `Potability = 1` vẫn còn hạn chế
do Recall thấp.

Test set vẫn chưa được sử dụng và sẽ được giữ lại cho bước đánh giá cuối cùng.

# Tổng kết Model Training - Thành viên 2

Trong notebook này, thành viên 2 đã thực hiện huấn luyện và đánh giá
hai mô hình:

1. Logistic Regression.
2. Support Vector Machine (SVM).

## Logistic Regression

Logistic Regression baseline có xu hướng dự đoán hầu hết mẫu về lớp
`Potability = 0`.

Sau Hyperparameter Tuning bằng GridSearchCV, cấu hình tốt nhất là:

- `C = 0.01`
- `class_weight = "balanced"`
- `solver = "liblinear"`

Kết quả Cross-Validation sau tuning:

- Accuracy: `0.4962`
- Precision: `0.3802`
- Recall: `0.4658`
- F1-score: `0.4183`
- ROC-AUC: `0.4766`

Việc sử dụng `class_weight="balanced"` giúp cải thiện mạnh Recall
và F1 so với baseline, tuy nhiên khả năng phân biệt hai lớp vẫn còn thấp.

## Support Vector Machine

SVM baseline với RBF Kernel đạt:

- Accuracy: `0.6798`
- Precision: `0.7190`
- Recall: `0.2935`
- F1-score: `0.4161`
- ROC-AUC: `0.7019`

SVM có Accuracy, Precision và ROC-AUC cao hơn Logistic Regression,
nhưng Recall thấp hơn, cho thấy mô hình vẫn bỏ sót nhiều mẫu nước
thuộc lớp `Potability = 1`.

Ngoài ra, khoảng cách giữa Train và Validation của SVM lớn hơn
Logistic Regression, cho thấy SVM có dấu hiệu overfitting rõ hơn.

## Quy trình chung

- Dataset được chia Train/Test theo tỷ lệ 80/20.
- `random_state = 42`.
- Sử dụng `stratify = y`.
- Missing values được xử lý bằng Median Imputation.
- Logistic Regression và SVM đều sử dụng StandardScaler.
- Preprocessing được đặt trong Pipeline nhằm hạn chế data leakage.
- Cross-Validation chỉ thực hiện trên training set.
- Test set chưa được sử dụng để lựa chọn model.

Bước tiếp theo là đánh giá các model trên test set bằng cùng bộ metric:

- Accuracy.
- Precision.
- Recall.
- F1-score.
- ROC-AUC.
- Confusion Matrix.

Kết quả test cuối cùng sẽ được sử dụng để so sánh Logistic Regression,
SVM, KNN và Random Forest.

## 6. Hyperparameter Tuning - SVM

SVM baseline cho kết quả ROC-AUC tương đối tốt nhưng Recall còn thấp
và có khoảng cách đáng kể giữa Train và Validation.

Do đó, GridSearchCV được sử dụng để tìm cấu hình phù hợp hơn.

Các hyperparameter được khảo sát:

- `C`: mức độ regularization.
- `gamma`: mức ảnh hưởng của từng điểm dữ liệu đối với RBF Kernel.
- `class_weight`: xem xét cân bằng hai lớp.

SVM tiếp tục sử dụng RBF Kernel.

Metric chính dùng để lựa chọn cấu hình là F1-score nhằm cân bằng
Precision và Recall của lớp `Potability = 1`.

Hyperparameter Tuning chỉ được thực hiện trên training set.
Test set vẫn được giữ riêng cho bước đánh giá cuối cùng.

In [51]:
svm_tuning_pipeline = Pipeline([
    (
        "preprocessor",
        build_scaled_preprocessor()
    ),
    (
        "classifier",
        SVC(
            kernel="rbf"
        )
    )
])

svm_tuning_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each 

In [52]:
svm_param_grid = {
    "classifier__C": [
        0.1,
        1.0,
        10.0
    ],
    "classifier__gamma": [
        "scale",
        0.01,
        0.1,
        1.0
    ],
    "classifier__class_weight": [
        None,
        "balanced"
    ]
}

svm_param_grid

{'classifier__C': [0.1, 1.0, 10.0],
 'classifier__gamma': ['scale', 0.01, 0.1, 1.0],
 'classifier__class_weight': [None, 'balanced']}

In [53]:
svm_grid_search = GridSearchCV(
    estimator=svm_tuning_pipeline,
    param_grid=svm_param_grid,
    scoring="f1",
    cv=cv_strategy,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

svm_grid_search

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...ier', SVC())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__C': [0.1, 1.0, ...], 'classifier__class_weight': [None, 'balanced'], 'classifier__gamma': ['scale', 0.01, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` an

In [54]:
svm_grid_search.fit(
    X_train,
    y_train
)

print("GridSearchCV SVM hoàn thành.")

Fitting 5 folds for each of 24 candidates, totalling 120 fits
GridSearchCV SVM hoàn thành.


In [56]:
print("Best parameters:")
print(svm_grid_search.best_params_)

print("\nBest CV F1:")
print(round(svm_grid_search.best_score_, 4))

Best parameters:
{'classifier__C': 1.0, 'classifier__class_weight': 'balanced', 'classifier__gamma': 'scale'}

Best CV F1:
0.5704


In [57]:
svm_grid_results = pd.DataFrame(
    svm_grid_search.cv_results_
)

svm_top_results = (
    svm_grid_results[
        [
            "param_classifier__C",
            "param_classifier__gamma",
            "param_classifier__class_weight",
            "mean_train_score",
            "mean_test_score",
            "std_test_score",
            "rank_test_score"
        ]
    ]
    .sort_values("rank_test_score")
    .head(10)
)

svm_top_results

,param_classifier__C,param_classifier__gamma,param_classifier__class_weight,mean_train_score,mean_test_score,std_test_score,rank_test_score
12,1.0,scale,balanced,0.696448,0.570371,0.029925,1
22,10.0,0.1,balanced,0.790645,0.567519,0.017370,2
14,1.0,0.1,balanced,0.681311,0.566076,0.030153,3
20,10.0,scale,balanced,0.809069,0.563867,0.016402,4
16,10.0,scale,NaN,0.782803,0.514587,0.031924,5
18,10.0,0.1,NaN,0.755368,0.503484,0.024537,6
4,0.1,scale,balanced,0.564685,0.501145,0.009526,7
6,0.1,0.1,balanced,0.553231,0.494153,0.009482,8
21,10.0,0.01,balanced,0.529388,0.469500,0.026399,9
8,1.0,scale,NaN,0.543808,0.416060,0.034419,10


In [58]:
best_svm_pipeline = svm_grid_search.best_estimator_

best_svm_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['ph','Hardness','Solids',...,'Organic_carbon','Trihalomethanes', 'Turbidity']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be insp

In [59]:
best_svm_cv_results = cross_validate(
    estimator=best_svm_pipeline,
    X=X_train,
    y=y_train,
    cv=cv_strategy,
    scoring=scoring_metrics,
    return_train_score=True
)

print("Cross-Validation best SVM hoàn thành.")

Cross-Validation best SVM hoàn thành.


In [60]:
best_svm_cv_summary = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC"
    ],
    "Mean": [
        best_svm_cv_results["test_accuracy"].mean(),
        best_svm_cv_results["test_precision"].mean(),
        best_svm_cv_results["test_recall"].mean(),
        best_svm_cv_results["test_f1"].mean(),
        best_svm_cv_results["test_roc_auc"].mean()
    ],
    "Std": [
        best_svm_cv_results["test_accuracy"].std(),
        best_svm_cv_results["test_precision"].std(),
        best_svm_cv_results["test_recall"].std(),
        best_svm_cv_results["test_f1"].std(),
        best_svm_cv_results["test_roc_auc"].std()
    ]
})

best_svm_cv_summary.round(4)

,Metric,Mean,Std
0,Accuracy,0.6668,0.0147
1,Precision,0.5733,0.0173
2,Recall,0.5694,0.0523
3,F1,0.5704,0.0299
4,ROC_AUC,0.7036,0.0217


In [61]:
best_svm_train_validation = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC"
    ],
    "Train_Mean": [
        best_svm_cv_results["train_accuracy"].mean(),
        best_svm_cv_results["train_precision"].mean(),
        best_svm_cv_results["train_recall"].mean(),
        best_svm_cv_results["train_f1"].mean(),
        best_svm_cv_results["train_roc_auc"].mean()
    ],
    "Validation_Mean": [
        best_svm_cv_results["test_accuracy"].mean(),
        best_svm_cv_results["test_precision"].mean(),
        best_svm_cv_results["test_recall"].mean(),
        best_svm_cv_results["test_f1"].mean(),
        best_svm_cv_results["test_roc_auc"].mean()
    ]
})

best_svm_train_validation["Gap"] = (
    best_svm_train_validation["Train_Mean"]
    - best_svm_train_validation["Validation_Mean"]
)

best_svm_train_validation.round(4)

,Metric,Train_Mean,Validation_Mean,Gap
0,Accuracy,0.7607,0.6668,0.0939
1,Precision,0.6893,0.5733,0.1160
2,Recall,0.7038,0.5694,0.1343
3,F1,0.6964,0.5704,0.1261
4,ROC_AUC,0.8242,0.7036,0.1206


### Kết luận Support Vector Machine (SVM)

SVM được xây dựng theo Pipeline:

`Median Imputation`
→ `StandardScaler`
→ `SVC`

SVM baseline sử dụng RBF Kernel với:

- `C = 1.0`
- `gamma = "scale"`

Kết quả Cross-Validation baseline:

- Accuracy: `0.6798 ± 0.0112`
- Precision: `0.7190 ± 0.0236`
- Recall: `0.2935 ± 0.0320`
- F1-score: `0.4161 ± 0.0344`
- ROC-AUC: `0.7019 ± 0.0179`

SVM baseline có Precision và ROC-AUC tương đối tốt nhưng Recall còn thấp,
nghĩa là mô hình vẫn bỏ sót nhiều mẫu thuộc lớp `Potability = 1`.

Sau đó, GridSearchCV được sử dụng để tuning các tham số `C`, `gamma`
và `class_weight`. Metric F1-score được sử dụng làm tiêu chí chính
để lựa chọn cấu hình.

Cấu hình SVM sau tuning được tiếp tục đánh giá bằng Stratified
5-Fold Cross-Validation trên training set với Accuracy, Precision,
Recall, F1-score và ROC-AUC.

Test set chưa được sử dụng trong toàn bộ quá trình training và tuning.
Test set được giữ riêng để đánh giá cuối cùng và so sánh công bằng
giữa Logistic Regression, SVM, KNN và Random Forest.